# 05 — Feature Engineering

## Objective

Transform raw smartphone specifications into normalized ML-ready features.

Steps:
1. Extract numeric features (RAM_GB, Storage_GB, Battery_mAh, Screen_Size_Inch, Refresh_Rate_Hz)
2. Compute subsystem scores (camera, performance, battery, display, AI, durability)
3. Compute final recommendation_score

**Input:** `data/processed/cleaned_dataset.csv`

**Output:** `data/processed/engineered_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np
import re
import sys
import os

# Add src to path for importing feature_engineering module

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Libraries imported successfully.")

In [ ]:
"""Feature engineering helpers for the recommendation system."""
import os
import re
import pandas as pd
import numpy as np

def extract_ram_gb(value):
    if pd.isna(value):
        return np.nan

    text = str(value)
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(TB|GB|MB)', text, flags=re.IGNORECASE)

    if not matches:
        return np.nan

    converted_values = []
    for number, unit in matches:
        numeric_value = float(number)
        unit = unit.upper()

        if unit == "TB":
            numeric_value *= 1024
        elif unit == "MB":
            numeric_value /= 1024

        converted_values.append(numeric_value)

    return max(converted_values)

def extract_storage_gb(value):
    if pd.isna(value):
        return np.nan

    text = str(value)
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(TB|GB|MB)', text, flags=re.IGNORECASE)

    if not matches:
        return np.nan

    converted_values = []
    for number, unit in matches:
        numeric_value = float(number)
        unit = unit.upper()

        if unit == "TB":
            numeric_value *= 1024
        elif unit == "MB":
            numeric_value /= 1024

        converted_values.append(numeric_value)

    return max(converted_values)

def extract_battery_mAh(value):
    if pd.isna(value):
        return np.nan

    text = str(value)
    match = re.search(r'(\d+(?:\.\d+)?)\s*mAh', text, flags=re.IGNORECASE)

    if match:
        return float(match.group(1))

    return np.nan

def extract_screen_size_inch(value):
    if pd.isna(value):
        return np.nan

    text = str(value)

    patterns = [
        r'(\d+(?:\.\d+)?)\s*(?:-|\s)?(?:inch|inches|in|\")',
        r'(\d+(?:\.\d+)?)\s*(?:-|\s)?(?:\u201d|\u201c)',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return float(match.group(1))

    return np.nan

def extract_refresh_rate_hz(value):
    if pd.isna(value):
        return 60.0

    text = str(value)
    match = re.search(r'(\d+)\s*Hz', text, flags=re.IGNORECASE)

    if match:
        return float(match.group(1))

    return 60.0

def normalize(series):
    numeric_series = pd.to_numeric(series, errors="coerce")
    normalized_series = pd.Series(5.0, index=series.index, dtype=float)

    valid_values = numeric_series.dropna()
    if valid_values.empty:
        return normalized_series

    minimum_value = valid_values.min()
    maximum_value = valid_values.max()

    if minimum_value == maximum_value:
        return normalized_series

    normalized_values = 10 * (numeric_series - minimum_value) / (maximum_value - minimum_value)
    normalized_values = normalized_values.clip(0, 10)
    normalized_values = normalized_values.fillna(5.0)

    return normalized_values.astype(float)

def encode_ois_flag(value):
    if pd.isna(value):
        return 0

    text = str(value).strip().lower()
    return int(text in {"yes", "true", "1", "y", "ois"})

def encode_ai_score(value):
    if pd.isna(value):
        return 0.0

    text = str(value).strip().lower()

    if "galaxy ai" in text:
        return 10.0
    if "basic ai" in text:
        return 5.0
    if text in {"no", "none", "no ai", "0", "", "na", "n/a"}:
        return 0.0

    return 0.0



In [ ]:
df = pd.read_csv("../data/processed/cleaned_dataset.csv")
print(f"Loaded dataset shape: {df.shape}")
print(f"Available columns: {list(df.columns)}")
df.head(3)

## 1. Extract Numeric Features

In [ ]:
    extract_ram_gb, extract_storage_gb, extract_battery_mAh,
    extract_screen_size_inch, extract_refresh_rate_hz
)

df["RAM_GB"] = df["RAM"].apply(extract_ram_gb)
df["Storage_GB"] = df["Storage"].apply(extract_storage_gb)
df["Battery_mAh"] = df["Battery"].apply(extract_battery_mAh)
df["Screen_Size_Inch"] = df["Display"].apply(extract_screen_size_inch)
df["Refresh_Rate_Hz"] = df["Display"].apply(extract_refresh_rate_hz)

print("Extracted numeric features:")
df[["Name", "RAM_GB", "Storage_GB", "Battery_mAh", "Screen_Size_Inch", "Refresh_Rate_Hz"]].head(10)

## 2. Impute Missing Values

In [ ]:
# Fill missing values with medians or defaults
for col, default in [("RAM_GB", 6.0), ("Storage_GB", 128.0), ("Battery_mAh", 4500.0),
                     ("Screen_Size_Inch", 6.5), ("Refresh_Rate_Hz", 60.0)]:
    median_val = df[col].median() if not df[col].isna().all() else default
    missing = df[col].isna().sum()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled {missing} missing values (median={median_val:.1f})")

print(f"\nRemaining NaN in numeric features: {df[['RAM_GB', 'Storage_GB', 'Battery_mAh', 'Screen_Size_Inch', 'Refresh_Rate_Hz']].isna().sum().sum()}")

## 3. Compute Subsystem Scores

In [ ]:

# Camera Score
df["camera_score"] = (
    0.60 * normalize(df["Main_Camera_MP"]) +
    0.20 * normalize(df["UltraWide_MP"]) +
    0.15 * normalize(df["Telephoto_MP"]) +
    0.05 * (df["OIS"].apply(encode_ois_flag) * 10)
)

# Performance Score
df["performance_score"] = (
    0.70 * normalize(df["RAM_GB"]) +
    0.30 * normalize(df["Storage_GB"])
)

# Battery Score
df["battery_score"] = normalize(df["Battery_mAh"])

# Display Score
df["display_score"] = (
    0.50 * normalize(df["Screen_Size_Inch"]) +
    0.50 * normalize(df["Refresh_Rate_Hz"])
)

# AI Score
df["ai_score"] = df["AI_Features"].apply(encode_ai_score)

# Durability Score
df["durability_score"] = df["Waterproof"].apply(
    lambda v: 10.0 if str(v).strip().lower() in {"yes", "true", "1", "y"} else 0.0
)

print("Subsystem scores computed.")
score_cols = ["camera_score", "performance_score", "battery_score", "display_score", "ai_score", "durability_score"]
df[score_cols].describe().round(2)

## 4. Compute Recommendation Score

**Formula:**

```
recommendation_score = 0.30 × performance + 0.25 × camera + 0.15 × battery + 0.15 × display + 0.10 × AI + 0.05 × durability
```

In [ ]:
# Compute raw recommendation score
recommendation_raw = (
    0.30 * df["performance_score"] +
    0.25 * df["camera_score"] +
    0.15 * df["battery_score"] +
    0.15 * df["display_score"] +
    0.10 * df["ai_score"] +
    0.05 * df["durability_score"]
)

# Normalize to 0-10
df["recommendation_score"] = normalize(recommendation_raw)

print("Recommendation score computed.")
print(f"Score range: {df['recommendation_score'].min():.2f} to {df['recommendation_score'].max():.2f}")
print(f"Mean: {df['recommendation_score'].mean():.2f}, Std: {df['recommendation_score'].std():.2f}")

## 5. Top & Bottom Phones by Recommendation Score

In [ ]:
display_cols = ["Name", "Series", "Launch_Year", "Launch_Price", "recommendation_score",
                "performance_score", "camera_score", "battery_score", "display_score", "ai_score"]

print("Top 10 Recommended Phones:")
print(df.nlargest(10, "recommendation_score")[display_cols].to_string(index=False))

print("\nBottom 10 Recommended Phones:")
print(df.nsmallest(10, "recommendation_score")[display_cols].to_string(index=False))

## 6. Save Engineered Dataset

In [ ]:
output_path = "../data/processed/engineered_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Engineered dataset saved to: {output_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")